In [8]:
import plotly.express as px
import datetime as dt
import pandas as pd
from lunarcalendar import Converter, Solar, Lunar, DateNotExist
from pandas_profiling import ProfileReport

In [9]:
import itertools
import datetime as dt
from sklearn.metrics import mean_squared_log_error, mean_squared_error

In [10]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 2000)
pd.set_option('display.max_colwidth', -1)

/mnt/GVO/space/revenue/env/lib/python3.6/site-packages/ipykernel_launcher.py:4: FutureWarning:

Passing a negative integer is deprecated in version 1.0 and will not be supported in future version. Instead, use None to not limit the column width.



In [11]:
%%capture
from tqdm import tqdm_notebook as tqdm
tqdm().pandas()


In [12]:
import pandas as pd
import numpy as np
import pandas as pd
from fbprophet import Prophet
import datetime as dt
from sklearn import metrics
import math
import itertools
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
from fbprophet.diagnostics import performance_metrics
from fbprophet.diagnostics import cross_validation

In [13]:

def mean_absolute_percentage_error(y_true, y_pred): 
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Read data 

In [7]:
revenue = pd.read_csv("/mnt/GVO/space/revenue/data/DT.csv", encoding='latin-1', sep = "\t")
# revenuet10 = pd.read_csv("DT_20201013.csv", encoding='latin-1', sep = "\t")
revenue = pd.concat([revenue, revenuet10])
revenue.columns = ["soluonghoadon", "soluongkhach", "doanhso", "doanhthu", "giamgiamon", "giamgiahoadon", "giamgiathanhvien", "vat", "phuphi", "macuahang", "tencuahang", "ngay" ]
revenue = revenue[["macuahang", "tencuahang", "ngay", "soluonghoadon", "soluongkhach", "doanhso", "giamgiamon", "giamgiahoadon", "giamgiathanhvien", "vat", "phuphi", "doanhthu"]]
revenue["ngay"] = pd.to_datetime(revenue['ngay'], format="%d/%m/%Y")
revenue["dow"] = revenue["ngay"].dt.dayofweek

FileNotFoundError: [Errno 2] No such file or directory: 'DT.csv'

# Prophet


In [401]:
def nfl_sunday(ds):
    date = ds
    if (date.weekday() == 5):
        return 1
    else:
        return 0


In [402]:
def nfl_monday(ds):
    date = ds
    if  (date.weekday() == 6)|(date.weekday() == 0):
        return 1
    else:
        return 0


In [431]:
independent = pd.DataFrame({
  'holiday': 'independent',
  'ds': pd.to_datetime(['2018-09-02', '2019-09-02', '2020-09-02']),
  'lower_window':-1,
  'upper_window': 0,
})

women = pd.DataFrame({
  'holiday': 'women',
  'ds': pd.to_datetime(['2018-03-08', '2018-10-20','2019-03-08', '2019-10-20','2019-11-20', '2020-03-08', '2020-10-20','2020-11-20']),
  'lower_window':-2,
  'upper_window': 0,
})

labor = pd.DataFrame({
  'holiday': 'labor',
  'ds': pd.to_datetime(['2018-04-30', '2018-05-01', '2019-04-30', '2019-05-01']),
  'lower_window':-3,
  'upper_window': 0,
})

xmas = pd.DataFrame({
  'holiday': 'xmas',
  'ds': pd.to_datetime(['2018-12-24', '2018-12-25', '2019-01-01', '2019-12-24', '2019-12-25','2020-01-01', '2020-12-24', '2020-12-25', '2021-01-01']),
  'lower_window':-3,
  'upper_window': 0,
})

holidays = pd.concat((independent, women, labor , xmas))

In [432]:
params = [[3, 5, 10, 0.5, 10, 0.5],[25, 25, 50, 0.1, 10, 0.1],[50, 30, 60, 0.3, 10, 0.3],[2, 4, 8, 0.8, 10, 0.8],
         [2, 2, 4, 0.8, 10, 0.8], [2, 3, 25, 0.1, 10, 0.1],[5, 2, 2, 0.5, 0.3, 0.01]]

param_grid = {  
    'wseas':[2,3,5,10,25,30,40,50],
    'mseas':[2],
    'yseas':[2,3,5,10,25,30,40,50],
    'seasonality_prior_scale': [0.01, 0.1, 0.3, 0.5, 1.0, 10.0],
    'holiday_prior_scale': [0.01, 0.1, 0.3, 0.5, 1.0, 10.0],
    'changepoint_prior_scale': [0.001, 0.01,0.05,0.1, 0.2, 0.3, 0.5],
    
    
}
all_params = [list(v) for v in itertools.product(*param_grid.values())]


In [433]:
def build_model(pars):
    wseas, mseas, yseas, s_prior, h_prior,c_prior= pars
    m = Prophet(growth = 'linear',
#                 seasonality_mode='multiplicative',
                holidays=holidays,
                changepoint_range = 0.9,
                daily_seasonality = False,
                weekly_seasonality = False,
                yearly_seasonality = False,
                holidays_prior_scale =h_prior,
                seasonality_prior_scale = s_prior,
                changepoint_prior_scale = c_prior
                   )
        


    m = m.add_seasonality(
        name = 'weekly',
        period=7,
        fourier_order = wseas)

#     m = m.add_seasonality(
#         name = 'monthly',
#         period=30.5,
#         fourier_order = mseas)


    m = m.add_seasonality(
        name = 'yearly',
        period=365.25,
        fourier_order = yseas)
    m.add_regressor('nfl_sunday')
#     m.add_regressor('nfl_monday')
    return m


In [434]:
def onecolfcst(current_col, params):  
    
    best_error = np.inf
    best_params = ()
    best_val_forecast = 0
    
    current_data = pd.DataFrame({
        'ds' : train.ngay,
        'y' : train[current_col],
        "nfl_sunday": train['nfl_sunday'],
        "nfl_monday": train['nfl_monday']
    })
    rmses = []
    for pars in tqdm(params):
        m = build_model(pars)  
        m.fit(current_data)
        df_cv = cross_validation(m, cutoffs=cutoffs, initial=500, horizon='30 days', parallel="processes")
        curerror = performance_metrics(df_cv, rolling_window=1)['mape'].values[0]
        print(pars, curerror)
        rmses.append(curerror)
    result = pd.DataFrame()
    result["params"] = params
    result["rmse"] = rmses
    return result

In [435]:
def predict_model(train, test, params, current_col):
    m = build_model(params)   
    current_data = pd.DataFrame({
        'ds' : list(train.ngay),
        'y' : list(train[current_col]),
        'nfl_sunday' : list(train["nfl_sunday"]),
        'nfl_monday' : list(train["nfl_monday"])})
    
    m.fit(current_data)
    
    predict_feature = pd.DataFrame({
            'ds': test.ngay,
            "nfl_sunday": test['nfl_sunday'],
            "nfl_monday": test['nfl_monday']
        })
    realforecast = m.predict(predict_feature)
    return list(realforecast['yhat'])

In [461]:
def predict_next(train, params, current_col, day = 30):
    m = build_model(params)   
    current_data = pd.DataFrame({
        'ds' : list(train.ngay),
        'y' : list(train[current_col]),
        'nfl_sunday' : list(train["nfl_sunday"]),
        'nfl_monday' : list(train["nfl_monday"])})
    
    m.fit(current_data)
    future = m.make_future_dataframe(periods = day,include_history =False)
    future["nfl_sunday"] = future.ds.apply(nfl_sunday)
    future["nfl_monday"] = future.ds.apply(nfl_monday)
    predict_feature = pd.DataFrame({
            'ds': future.ds,
            "nfl_sunday": future['nfl_sunday'],
            "nfl_monday": future['nfl_monday']
        })
    realforecast = m.predict(predict_feature)
    future["predict"] = list(realforecast['yhat'])
    return future

In [437]:
cutoffs = pd.to_datetime(['2020-07-31'])

In [438]:
# store = revenue[revenue.macuahang.isin([1,4,5,6,16,18])]
# store = store[(store.ngay< dt.datetime(2020,3,6))|(store.ngay> dt.datetime(2020,6,16))]
# store = store[store.doanhso!=0]

In [689]:
store = revenue[revenue.macuahang.isin([1,4,5,6,16,18,19])]
store = store[(store.ngay< dt.datetime(2020,3,6))|(store.ngay> dt.datetime(2020,6,16))]
store = store[store.doanhso!=0]
store =store[(store.macuahang!=19)|((store.macuahang==19)&(store.ngay> dt.datetime(2018,2,3)))]

## Ung van khiem

In [690]:
ungvankhiem = store[store.macuahang==1]

In [691]:
ungvankhiem

,macuahang,tencuahang,ngay,soluonghoadon,soluongkhach,doanhso,giamgiamon,giamgiahoadon,giamgiathanhvien,vat,phuphi,doanhthu,dow
0,1,ICOOL Ung V?n Khiêm,2017-07-25,70,418,86496200.0,923600.0,472120.0,2729705.0,402623.5,0,82370775.0,1
7,1,ICOOL Ung V?n Khiêm,2017-07-26,75,481,95504100.0,356000.0,6417750.0,2010085.0,0.0,0,86720265.0,2
14,1,ICOOL Ung V?n Khiêm,2017-07-27,76,418,96707700.0,759000.0,1505440.0,1920390.0,119583.0,500000,92522870.0,3
21,1,ICOOL Ung V?n Khiêm,2017-07-28,87,596,128260700.0,3134300.0,1105580.0,2474980.0,293317.5,400000,121545840.0,4
28,1,ICOOL Ung V?n Khiêm,2017-07-29,109,768,158043700.0,1419000.0,432440.0,3837745.0,133960.0,0,152354515.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
34,1,ICool Ung V?n Khiêm,2020-10-20,105,723,178692934.0,6492683.3,0.0,121480.0,652490.0,0,172078770.7,1
35,1,ICool Ung V?n Khiêm,2020-10-21,60,379,82038000.0,2917470.0,1654000.0,0.0,231312.0,0,77466530.0,2
36,1,ICool Ung V?n Khiêm,2020-10-22,66,426,83757400.0,2746320.0,573330.0,0.0,336595.0,0,80437750.0,3
37,1,ICool Ung V?n Khiêm,2020-10-23,73,504,107279801.0,4338130.0,1612000.0,0.0,266907.5,0,101329671.0,4


In [692]:
firstday = dt.datetime.strptime('2019/1/1', '%Y/%m/%d').date()
lastday = dt.datetime.strptime('2020/9/30', '%Y/%m/%d').date() 

In [693]:
date_period = pd.date_range(firstday, lastday- dt.timedelta(days = 61))
date_period1 = pd.date_range(lastday- dt.timedelta(days = 60), lastday- dt.timedelta(days = 30))
date_period2 = pd.date_range(lastday- dt.timedelta(days = 29), lastday)

In [694]:
all_prediod =  pd.date_range(firstday, lastday)
date_test_period =  pd.date_range(dt.datetime.strptime('2020/10/1', '%Y/%m/%d').date(), dt.datetime.strptime('2020/10/31', '%Y/%m/%d').date() )
date_11 =  pd.date_range(dt.datetime.strptime('2020/11/1', '%Y/%m/%d').date(), dt.datetime.strptime('2020/11/20', '%Y/%m/%d').date() )

In [695]:
date_df =pd.DataFrame(pd.date_range(firstday, dt.datetime.strptime('2020/11/20', '%Y/%m/%d').date() ), columns = ["ngay"])
data_df = date_df.merge(ungvankhiem[["ngay", "doanhthu"]], on = "ngay", how = "left")
# date_df.comlumns

In [696]:
data_df["nfl_sunday"] = data_df['ngay'].apply(nfl_sunday)
data_df["nfl_monday"] = data_df['ngay'].apply(nfl_monday)

In [697]:
train = data_df[data_df.ngay.isin(list(date_period)+list(date_period1) + list(date_period2))]
test = data_df[data_df.ngay.isin(date_period2)].reset_index().drop(["index"], axis=1)

In [7]:
all_data = data_df[data_df.ngay.isin(all_prediod)].reset_index().drop(["index"], axis=1)
all_data_10 = data_df[data_df.ngay.isin(pd.date_range(firstday,dt.datetime.strptime('2020/10/31', '%Y/%m/%d').date()))].reset_index().drop(["index"], axis=1)
backtest = data_df[data_df.ngay.isin(date_test_period)].reset_index().drop(["index"], axis=1)
backtest_11 = data_df[data_df.ngay.isin(date_11)].reset_index().drop(["index"], axis=1)

NameError: name 'data_df' is not defined

In [699]:
error = onecolfcst("doanhthu", params = params)

 14%|█▍        | 1/7 [00:02<00:12,  2.09s/it]

[3, 5, 10, 0.5, 10, 0.5] 0.3681795165106272


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f248f2c7cf8>
 29%|██▊       | 2/7 [00:04<00:11,  2.28s/it]

[25, 25, 50, 0.1, 10, 0.1] 0.37336237098525416


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f24534294a8>
 43%|████▎     | 3/7 [00:07<00:09,  2.40s/it]

...

[2, 2, 4, 0.8, 10, 0.8] 0.29924021718865235


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f24534f2828>
 86%|████████▌ | 6/7 [00:14<00:02,  2.20s/it]INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f2454ea1a58>


[2, 3, 25, 0.1, 10, 0.1] 0.36509429784751996


100%|██████████| 7/7 [00:15<00:00,  2.26s/it]

[5, 2, 2, 0.5, 0.3, 0.01] 0.3214040640875949


In [703]:
best_params = [2, 2, 4, 0.8, 10, 0.8]

In [704]:
# best_params = error[error.rmse == error.rmse.min()].params.values[0]
backtest["predict"] = predict_model(all_data, backtest,best_params, "doanhthu")

In [705]:
fig = px.line(backtest, x="ngay", y=["doanhthu","predict"])
fig.show()

In [706]:
print("Doanh thu thuc te", backtest.doanhthu.sum())
print("Doanh thu du doan", backtest.predict.sum())
diff_percent = (backtest.doanhthu.sum() - backtest.predict.sum())/backtest.doanhthu.sum()
print("diff_percent", diff_percent)
print("mape", mean_absolute_percentage_error(backtest.doanhthu, backtest.predict))
print("rmse", mean_squared_error(backtest.doanhthu, backtest.predict, squared=False))

Doanh thu thuc te 3660728234.1
Doanh thu du doan 3657480367.829054
diff_percent 0.0008872186251609367
mape 12.963310262118977
rmse 17541109.557437077


In [707]:
ungvankhiem_10 = backtest[["ngay", "predict"]]
ungvankhiem_10.columns = ["ngay", "ungvankhiem"]

In [710]:
backtest_11["predict"] = predict_model(all_data_10, backtest_11,best_params, "doanhthu")
(backtest_11.doanhthu.sum() - backtest_11.predict.sum())/backtest_11.doanhthu.sum()

0.08851448331237798

In [711]:
print("Doanh thu thuc te", backtest_11.doanhthu.sum())
print("Doanh thu du doan", backtest_11.predict.sum())
diff_percent = (backtest_11.doanhthu.sum() - backtest_11.predict.sum())/backtest_11.doanhthu.sum()
print("diff_percent", diff_percent)
print("mape", mean_absolute_percentage_error(backtest_11.doanhthu, backtest_11.predict))
print("rmse", mean_squared_error(backtest_11.doanhthu, backtest_11.predict, squared=False))

Doanh thu thuc te 2354301757.8
Doanh thu du doan 2145911954.14691
diff_percent 0.08851448331237798
mape 10.868273056645721
rmse 15998973.811941458


In [712]:
ungvankhiem_20_11 = backtest_11[["ngay", "predict"]]
ungvankhiem_20_11.columns = ["ngay", "ungvankhiem"]

In [713]:
fig = px.line(backtest_11, x="ngay", y=["doanhthu","predict"])
fig.show()

In [714]:
ungvankhiem_12 = predict_next(data_df,best_params,"doanhthu",41)

In [715]:
ungvankhiem_12 = ungvankhiem_12[["ds", "predict"]]
ungvankhiem_12.columns = ["ds", "ungvankhiem"]

## XVNT

In [716]:
xvnt = store[store.macuahang==4]

firstday = dt.datetime.strptime('2019/1/1', '%Y/%m/%d').date()
lastday = dt.datetime.strptime('2020/9/30', '%Y/%m/%d').date() 

date_period = pd.date_range(firstday, lastday- dt.timedelta(days = 61))
date_period1 = pd.date_range(lastday- dt.timedelta(days = 60), lastday- dt.timedelta(days = 30))
date_period2 = pd.date_range(lastday- dt.timedelta(days = 29), lastday)

all_prediod =  pd.date_range(firstday, lastday)
date_test_period =  pd.date_range(dt.datetime.strptime('2020/10/1', '%Y/%m/%d').date(), dt.datetime.strptime('2020/10/31', '%Y/%m/%d').date() )
date_11 =  pd.date_range(dt.datetime.strptime('2020/11/1', '%Y/%m/%d').date(), dt.datetime.strptime('2020/11/20', '%Y/%m/%d').date() )

date_df =pd.DataFrame(pd.date_range(firstday, dt.datetime.strptime('2020/11/20', '%Y/%m/%d').date() ), columns = ["ngay"])
data_df = date_df.merge(xvnt[["ngay", "doanhthu"]], on = "ngay", how = "left")
# date_df.comlumns

data_df["nfl_sunday"] = data_df['ngay'].apply(nfl_sunday)
data_df["nfl_monday"] = data_df['ngay'].apply(nfl_monday)

train = data_df[data_df.ngay.isin(list(date_period)+list(date_period1) + list(date_period2))]
test = data_df[data_df.ngay.isin(date_period2)].reset_index().drop(["index"], axis=1)

all_data = data_df[data_df.ngay.isin(all_prediod)].reset_index().drop(["index"], axis=1)
all_data_10 = data_df[data_df.ngay.isin(pd.date_range(firstday,dt.datetime.strptime('2020/10/31', '%Y/%m/%d').date()))].reset_index().drop(["index"], axis=1)
backtest = data_df[data_df.ngay.isin(date_test_period)].reset_index().drop(["index"], axis=1)
backtest_11 = data_df[data_df.ngay.isin(date_11)].reset_index().drop(["index"], axis=1)

error = onecolfcst("doanhthu", params = params)

best_params = [6, 3, 6, 0.8, 9, 0.8]

# best_params = error[error.rmse == error.rmse.min()].params.values[0]
backtest["predict"] = predict_model(all_data, backtest,best_params, "doanhthu")

fig = px.line(backtest, x="ngay", y=["doanhthu","predict"])
fig.show()

print("Doanh thu thuc te", backtest.doanhthu.sum())
print("Doanh thu du doan", backtest.predict.sum())
diff_percent = (backtest.doanhthu.sum() - backtest.predict.sum())/backtest.doanhthu.sum()
print("diff_percent", diff_percent)
print("mape", mean_absolute_percentage_error(backtest.doanhthu, backtest.predict))
print("rmse", mean_squared_error(backtest.doanhthu, backtest.predict, squared=False))

xvnt_10 = backtest[["ngay", "predict"]]
xvnt_10.columns = ["ngay", "xvnt"]

backtest_11["predict"] = predict_model(all_data_10, backtest_11,best_params, "doanhthu")
(backtest_11.doanhthu.sum() - backtest_11.predict.sum())/backtest_11.doanhthu.sum()

print("Doanh thu thuc te", backtest_11.doanhthu.sum())
print("Doanh thu du doan", backtest_11.predict.sum())
diff_percent = (backtest_11.doanhthu.sum() - backtest_11.predict.sum())/backtest_11.doanhthu.sum()
print("diff_percent", diff_percent)
print("mape", mean_absolute_percentage_error(backtest_11.doanhthu, backtest_11.predict))
print("rmse", mean_squared_error(backtest_11.doanhthu, backtest_11.predict, squared=False))

xvnt_20_11 = backtest_11[["ngay", "predict"]]
xvnt_20_11.columns = ["ngay", "xvnt"]

fig = px.line(backtest_11, x="ngay", y=["doanhthu","predict"])
fig.show()



xvnt_12 = predict_next(data_df,best_params,"doanhthu",41)

xvnt_12 = xvnt_12[["ds", "predict"]]
xvnt_12.columns = ["ds", "xvnt"]

## TOKY

In [739]:
toky = store[store.macuahang==5]

In [740]:
firstday = dt.datetime.strptime('2019/1/1', '%Y/%m/%d').date()
lastday = dt.datetime.strptime('2020/9/30', '%Y/%m/%d').date() 

In [741]:
date_period = pd.date_range(firstday, lastday- dt.timedelta(days = 61))
date_period1 = pd.date_range(lastday- dt.timedelta(days = 60), lastday- dt.timedelta(days = 30))
date_period2 = pd.date_range(lastday- dt.timedelta(days = 29), lastday)

In [742]:
all_prediod =  pd.date_range(firstday, lastday)
date_test_period =  pd.date_range(dt.datetime.strptime('2020/10/1', '%Y/%m/%d').date(), dt.datetime.strptime('2020/10/31', '%Y/%m/%d').date() )
date_11 =  pd.date_range(dt.datetime.strptime('2020/11/1', '%Y/%m/%d').date(), dt.datetime.strptime('2020/11/20', '%Y/%m/%d').date() )

In [743]:
date_df =pd.DataFrame(pd.date_range(firstday, dt.datetime.strptime('2020/11/20', '%Y/%m/%d').date() ), columns = ["ngay"])
data_df = date_df.merge(toky[["ngay", "doanhthu"]], on = "ngay", how = "left")
# date_df.comlumns

In [744]:
data_df["nfl_sunday"] = data_df['ngay'].apply(nfl_sunday)
data_df["nfl_monday"] = data_df['ngay'].apply(nfl_monday)

In [745]:
train = data_df[data_df.ngay.isin(list(date_period)+list(date_period1) + list(date_period2))]
test = data_df[data_df.ngay.isin(date_period2)].reset_index().drop(["index"], axis=1)

In [746]:
all_data = data_df[data_df.ngay.isin(all_prediod)].reset_index().drop(["index"], axis=1)
all_data_10 = data_df[data_df.ngay.isin(pd.date_range(firstday,dt.datetime.strptime('2020/10/31', '%Y/%m/%d').date()))].reset_index().drop(["index"], axis=1)
backtest = data_df[data_df.ngay.isin(date_test_period)].reset_index().drop(["index"], axis=1)
backtest_11 = data_df[data_df.ngay.isin(date_11)].reset_index().drop(["index"], axis=1)

In [492]:
error = onecolfcst("doanhthu", params = params)

 14%|█▍        | 1/7 [00:01<00:11,  1.97s/it]

[3, 5, 10, 0.5, 10, 0.5] 0.3288963331790131


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f24534442e8>
 29%|██▊       | 2/7 [00:04<00:10,  2.10s/it]

[25, 25, 50, 0.1, 10, 0.1] 0.4230368683878393


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f245341c518>
 43%|████▎     | 3/7 [00:07<00:09,  2.32s/it]

...

[2, 2, 4, 0.8, 10, 0.8] 0.29214872252654467


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f24534a2ac8>
 86%|████████▌ | 6/7 [00:13<00:02,  2.14s/it]INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f24534a6518>


[2, 3, 25, 0.1, 10, 0.1] 0.4319884656554507


100%|██████████| 7/7 [00:15<00:00,  2.18s/it]

[5, 2, 2, 0.5, 0.3, 0.01] 0.38341277733303336


In [747]:
best_params = [2, 3, 2, 0.8, 10, 0.8]

In [748]:
# best_params = error[error.rmse == error.rmse.min()].params.values[0]
backtest["predict"] = predict_model(all_data, backtest,best_params, "doanhthu")

In [749]:
fig = px.line(backtest, x="ngay", y=["doanhthu","predict"])
fig.show()

In [750]:
print("Doanh thu thuc te", backtest.doanhthu.sum())
print("Doanh thu du doan", backtest.predict.sum())
diff_percent = (backtest.doanhthu.sum() - backtest.predict.sum())/backtest.doanhthu.sum()
print("diff_percent", diff_percent)
print("mape", mean_absolute_percentage_error(backtest.doanhthu, backtest.predict))
print("rmse", mean_squared_error(backtest.doanhthu, backtest.predict, squared=False))

Doanh thu thuc te 1153399463.5
Doanh thu du doan 1187824777.2919421
diff_percent -0.02984682660374943
mape 32.333125472032485
rmse 10827059.830142789


In [751]:
backtest_11["predict"] = predict_model(all_data_10, backtest_11,best_params, "doanhthu")
(backtest_11.doanhthu.sum() - backtest_11.predict.sum())/backtest_11.doanhthu.sum()

0.04845976762369945

In [752]:
print("Doanh thu thuc te", backtest_11.doanhthu.sum())
print("Doanh thu du doan", backtest_11.predict.sum())
diff_percent = (backtest_11.doanhthu.sum() - backtest_11.predict.sum())/backtest_11.doanhthu.sum()
print("diff_percent", diff_percent)
print("mape", mean_absolute_percentage_error(backtest_11.doanhthu, backtest_11.predict))
print("rmse", mean_squared_error(backtest_11.doanhthu, backtest_11.predict, squared=False))

Doanh thu thuc te 772723209.8
Doanh thu du doan 735277222.6156528
diff_percent 0.04845976762369945
mape 18.230790113975726
rmse 9997556.674802288


In [753]:
fig = px.line(backtest_11, x="ngay", y=["doanhthu","predict"])
fig.show()

In [754]:
toky_10 = backtest[["ngay", "predict"]]
toky_10.columns = ["ngay", "toky"]
toky_20_11 = backtest_11[["ngay", "predict"]]
toky_20_11.columns = ["ngay", "toky"]

In [755]:
toky_12 = predict_next(data_df,best_params,"doanhthu",41)

In [756]:
toky_12 = toky_12[["ds", "predict"]]
toky_12.columns = ["ds", "toky"]

## binhphu

In [757]:
binhphu = store[store.macuahang==6]

In [758]:
firstday = dt.datetime.strptime('2019/1/1', '%Y/%m/%d').date()
lastday = dt.datetime.strptime('2020/9/30', '%Y/%m/%d').date() 

In [759]:
date_period = pd.date_range(firstday, lastday- dt.timedelta(days = 61))
date_period1 = pd.date_range(lastday- dt.timedelta(days = 60), lastday- dt.timedelta(days = 30))
date_period2 = pd.date_range(lastday- dt.timedelta(days = 29), lastday)

In [760]:
all_prediod =  pd.date_range(firstday, lastday)
date_test_period =  pd.date_range(dt.datetime.strptime('2020/10/1', '%Y/%m/%d').date(), dt.datetime.strptime('2020/10/31', '%Y/%m/%d').date() )
date_11 =  pd.date_range(dt.datetime.strptime('2020/11/1', '%Y/%m/%d').date(), dt.datetime.strptime('2020/11/20', '%Y/%m/%d').date() )

In [761]:
date_df =pd.DataFrame(pd.date_range(firstday, dt.datetime.strptime('2020/11/20', '%Y/%m/%d').date() ), columns = ["ngay"])
data_df = date_df.merge(binhphu[["ngay", "doanhthu"]], on = "ngay", how = "left")
# date_df.comlumns

In [762]:
data_df["nfl_sunday"] = data_df['ngay'].apply(nfl_sunday)
data_df["nfl_monday"] = data_df['ngay'].apply(nfl_monday)

In [763]:
train = data_df[data_df.ngay.isin(list(date_period)+list(date_period1) + list(date_period2))]
test = data_df[data_df.ngay.isin(date_period2)].reset_index().drop(["index"], axis=1)

In [764]:
all_data = data_df[data_df.ngay.isin(all_prediod)].reset_index().drop(["index"], axis=1)
all_data_10 = data_df[data_df.ngay.isin(pd.date_range(firstday,dt.datetime.strptime('2020/10/31', '%Y/%m/%d').date()))].reset_index().drop(["index"], axis=1)
backtest = data_df[data_df.ngay.isin(date_test_period)].reset_index().drop(["index"], axis=1)
backtest_11 = data_df[data_df.ngay.isin(date_11)].reset_index().drop(["index"], axis=1)

In [510]:
error = onecolfcst("doanhthu", params = params)

 14%|█▍        | 1/7 [00:02<00:14,  2.42s/it]

[3, 5, 10, 0.5, 10, 0.5] 0.3093012318096393


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f24553edc88>
 29%|██▊       | 2/7 [00:05<00:12,  2.47s/it]

[25, 25, 50, 0.1, 10, 0.1] 0.28563861743330393


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f24534a6550>
 43%|████▎     | 3/7 [00:09<00:12,  3.15s/it]

...

[2, 2, 4, 0.8, 10, 0.8] 0.25516640028398996


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f24534a60f0>
 86%|████████▌ | 6/7 [00:16<00:02,  2.50s/it]INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f245344e240>


[2, 3, 25, 0.1, 10, 0.1] 0.3025328295807638


100%|██████████| 7/7 [00:18<00:00,  2.58s/it]

[5, 2, 2, 0.5, 0.3, 0.01] 0.3484840392911539


In [765]:
best_params = [2, 3, 2, 0.8, 9, 0.8]

In [766]:
# best_params = error[error.rmse == error.rmse.min()].params.values[0]
backtest["predict"] = predict_model(all_data, backtest,best_params, "doanhthu")

In [767]:
fig = px.line(backtest, x="ngay", y=["doanhthu","predict"])
fig.show()

In [768]:
print("Doanh thu thuc te", backtest.doanhthu.sum())
print("Doanh thu du doan", backtest.predict.sum())
diff_percent = (backtest.doanhthu.sum() - backtest.predict.sum())/backtest.doanhthu.sum()
print("diff_percent", diff_percent)
print("mape", mean_absolute_percentage_error(backtest.doanhthu, backtest.predict))
print("rmse", mean_squared_error(backtest.doanhthu, backtest.predict, squared=False))

Doanh thu thuc te 1954257674.0
Doanh thu du doan 2122068681.9789183
diff_percent -0.08586943790039753
mape 27.27176464468002
rmse 15822946.450290486


In [769]:
backtest_11["predict"] = predict_model(all_data_10, backtest_11,best_params, "doanhthu")
(backtest_11.doanhthu.sum() - backtest_11.predict.sum())/backtest_11.doanhthu.sum()

-0.03137346661192275

In [770]:
print("Doanh thu thuc te", backtest_11.doanhthu.sum())
print("Doanh thu du doan", backtest_11.predict.sum())
diff_percent = (backtest_11.doanhthu.sum() - backtest_11.predict.sum())/backtest_11.doanhthu.sum()
print("diff_percent", diff_percent)
print("mape", mean_absolute_percentage_error(backtest_11.doanhthu, backtest_11.predict))
print("rmse", mean_squared_error(backtest_11.doanhthu, backtest_11.predict, squared=False))

Doanh thu thuc te 1224589595.45
Doanh thu du doan 1263009216.2361586
diff_percent -0.03137346661192275
mape 16.49884081637662
rmse 10849882.10982869


In [771]:
fig = px.line(backtest_11, x="ngay", y=["doanhthu","predict"])
fig.show()

In [772]:
binhphu_10 = backtest[["ngay", "predict"]]
binhphu_10.columns = ["ngay", "binhphu"]
binhphu_20_11 = backtest_11[["ngay", "predict"]]
binhphu_20_11.columns = ["ngay", "binhphu"]

In [773]:
binhphu_12 = predict_next(data_df,best_params,"doanhthu",41)

In [774]:
binhphu_12 = binhphu_12[["ds", "predict"]]
binhphu_12.columns = ["ds", "binhphu"]

## cauchuy

In [775]:
cauchuy = store[store.macuahang==16]

In [776]:
firstday = dt.datetime.strptime('2019/1/1', '%Y/%m/%d').date()
lastday = dt.datetime.strptime('2020/9/30', '%Y/%m/%d').date() 

In [777]:
date_period = pd.date_range(firstday, lastday- dt.timedelta(days = 61))
date_period1 = pd.date_range(lastday- dt.timedelta(days = 60), lastday- dt.timedelta(days = 30))
date_period2 = pd.date_range(lastday- dt.timedelta(days = 29), lastday)

In [778]:
all_prediod =  pd.date_range(firstday, lastday)
date_test_period =  pd.date_range(dt.datetime.strptime('2020/10/1', '%Y/%m/%d').date(), dt.datetime.strptime('2020/10/31', '%Y/%m/%d').date() )
date_11 =  pd.date_range(dt.datetime.strptime('2020/11/1', '%Y/%m/%d').date(), dt.datetime.strptime('2020/11/20', '%Y/%m/%d').date() )

In [779]:
date_df =pd.DataFrame(pd.date_range(firstday, dt.datetime.strptime('2020/11/20', '%Y/%m/%d').date() ), columns = ["ngay"])
data_df = date_df.merge(cauchuy[["ngay", "doanhthu"]], on = "ngay", how = "left")
# date_df.comlumns

In [780]:
data_df["nfl_sunday"] = data_df['ngay'].apply(nfl_sunday)
data_df["nfl_monday"] = data_df['ngay'].apply(nfl_monday)

In [781]:
train = data_df[data_df.ngay.isin(list(date_period)+list(date_period1) + list(date_period2))]
test = data_df[data_df.ngay.isin(date_period2)].reset_index().drop(["index"], axis=1)

In [782]:
all_data = data_df[data_df.ngay.isin(all_prediod)].reset_index().drop(["index"], axis=1)
all_data_10 = data_df[data_df.ngay.isin(pd.date_range(firstday,dt.datetime.strptime('2020/10/31', '%Y/%m/%d').date()))].reset_index().drop(["index"], axis=1)
backtest = data_df[data_df.ngay.isin(date_test_period)].reset_index().drop(["index"], axis=1)
backtest_11 = data_df[data_df.ngay.isin(date_11)].reset_index().drop(["index"], axis=1)

In [573]:
error = onecolfcst("doanhthu", params = params)

 14%|█▍        | 1/7 [00:01<00:11,  1.96s/it]

[3, 5, 10, 0.5, 10, 0.5] 0.42328484132715166


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f2455094e80>
 29%|██▊       | 2/7 [00:04<00:10,  2.14s/it]

[25, 25, 50, 0.1, 10, 0.1] 0.45689029610654625


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f2455618470>
 43%|████▎     | 3/7 [00:07<00:09,  2.36s/it]

...

[2, 2, 4, 0.8, 10, 0.8] 0.47351943461898205


 86%|████████▌ | 6/7 [00:13<00:02,  2.10s/it]INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f2455b31cf8>


[2, 3, 25, 0.1, 10, 0.1] 0.5266048645308966


100%|██████████| 7/7 [00:15<00:00,  2.15s/it]

[5, 2, 2, 0.5, 0.3, 0.01] 0.5065200882674072


In [783]:
best_params = [2, 3, 2, 0.8, 9, 0.8]

In [784]:
# best_params = error[error.rmse == error.rmse.min()].params.values[0]
backtest["predict"] = predict_model(all_data, backtest,best_params, "doanhthu")

In [785]:
fig = px.line(backtest, x="ngay", y=["doanhthu","predict"])
fig.show()

In [786]:
print("Doanh thu thuc te", backtest.doanhthu.sum())
print("Doanh thu du doan", backtest.predict.sum())
diff_percent = (backtest.doanhthu.sum() - backtest.predict.sum())/backtest.doanhthu.sum()
print("diff_percent", diff_percent)
print("mape", mean_absolute_percentage_error(backtest.doanhthu, backtest.predict))
print("rmse", mean_squared_error(backtest.doanhthu, backtest.predict, squared=False))

Doanh thu thuc te 397004648.0
Doanh thu du doan 357847165.98214704
diff_percent 0.09863230119626448
mape 31.42535773939467


ValueError: Input contains NaN, infinity or a value too large for dtype('float64').

In [787]:
backtest_11["predict"] = predict_model(all_data_10, backtest_11,best_params, "doanhthu")
(backtest_11.doanhthu.sum() - backtest_11.predict.sum())/backtest_11.doanhthu.sum()

-0.040235786161102356

In [788]:
# print("Doanh thu thuc te", backtest_11.doanhthu.sum())
# print("Doanh thu du doan", backtest_11.predict.sum())
# diff_percent = (backtest_11.doanhthu.sum() - backtest_11.predict.sum())/backtest_11.doanhthu.sum()
# print("diff_percent", diff_percent)
# print("mape", mean_absolute_percentage_error(backtest_11.doanhthu, backtest_11.predict))
# print("rmse", mean_squared_error(backtest_11.doanhthu, backtest_11.predict, squared=False))

In [789]:
fig = px.line(backtest_11, x="ngay", y=["doanhthu","predict"])
fig.show()

In [790]:
cauchuY_10 = backtest[["ngay", "predict"]]
cauchuY_10.columns = ["ngay", "cauchuY"]
cauchuY_20_11 = backtest_11[["ngay", "predict"]]
cauchuY_20_11.columns = ["ngay", "cauchuY"]

In [791]:
cauchuY_12 = predict_next(data_df,best_params,"doanhthu",41)

In [792]:
cauchuY_12 = cauchuY_12[["ds", "predict"]]
cauchuY_12.columns = ["ds", "cauchuY"]

## TranNao

In [793]:
tranao = store[store.macuahang==18]

In [794]:
firstday = dt.datetime.strptime('2019/1/1', '%Y/%m/%d').date()
lastday = dt.datetime.strptime('2020/9/30', '%Y/%m/%d').date() 

In [795]:
date_period = pd.date_range(firstday, lastday- dt.timedelta(days = 61))
date_period1 = pd.date_range(lastday- dt.timedelta(days = 60), lastday- dt.timedelta(days = 30))
date_period2 = pd.date_range(lastday- dt.timedelta(days = 29), lastday)

In [796]:
all_prediod =  pd.date_range(firstday, lastday)
date_test_period =  pd.date_range(dt.datetime.strptime('2020/10/1', '%Y/%m/%d').date(), dt.datetime.strptime('2020/10/31', '%Y/%m/%d').date() )
date_11 =  pd.date_range(dt.datetime.strptime('2020/11/1', '%Y/%m/%d').date(), dt.datetime.strptime('2020/11/20', '%Y/%m/%d').date() )

In [797]:
date_df =pd.DataFrame(pd.date_range(firstday, dt.datetime.strptime('2020/11/20', '%Y/%m/%d').date() ), columns = ["ngay"])
data_df = date_df.merge(tranao[["ngay", "doanhthu"]], on = "ngay", how = "left")
# date_df.comlumns

In [798]:
data_df["nfl_sunday"] = data_df['ngay'].apply(nfl_sunday)
data_df["nfl_monday"] = data_df['ngay'].apply(nfl_monday)

In [799]:
train = data_df[data_df.ngay.isin(list(date_period)+list(date_period1) + list(date_period2))]
test = data_df[data_df.ngay.isin(date_period2)].reset_index().drop(["index"], axis=1)

In [800]:
all_data = data_df[data_df.ngay.isin(all_prediod)].reset_index().drop(["index"], axis=1)
all_data_10 = data_df[data_df.ngay.isin(pd.date_range(firstday,dt.datetime.strptime('2020/10/31', '%Y/%m/%d').date()))].reset_index().drop(["index"], axis=1)
backtest = data_df[data_df.ngay.isin(date_test_period)].reset_index().drop(["index"], axis=1)
backtest_11 = data_df[data_df.ngay.isin(date_11)].reset_index().drop(["index"], axis=1)

In [548]:
error = onecolfcst("doanhthu", params = params)

 14%|█▍        | 1/7 [00:02<00:12,  2.04s/it]

[3, 5, 10, 0.5, 10, 0.5] 0.31821894700374986


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f24534fb7b8>
 29%|██▊       | 2/7 [00:04<00:10,  2.18s/it]

[25, 25, 50, 0.1, 10, 0.1] 0.29048845599261147


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f24532c9e80>
 43%|████▎     | 3/7 [00:07<00:09,  2.41s/it]

...

[2, 2, 4, 0.8, 10, 0.8] 0.2573732844259849


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f24534df780>
 86%|████████▌ | 6/7 [00:13<00:02,  2.18s/it]INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f24533e8ac8>


[2, 3, 25, 0.1, 10, 0.1] 0.27574516754615414


100%|██████████| 7/7 [00:15<00:00,  2.23s/it]

[5, 2, 2, 0.5, 0.3, 0.01] 0.1843378025516163


In [801]:
best_params = [2, 3, 2, 0.6, 5, 0.7]

In [802]:
# best_params = error[error.rmse == error.rmse.min()].params.values[0]
backtest["predict"] = predict_model(all_data, backtest,best_params, "doanhthu")

In [803]:
fig = px.line(backtest, x="ngay", y=["doanhthu","predict"])
fig.show()

In [804]:
print("Doanh thu thuc te", backtest.doanhthu.sum())
print("Doanh thu du doan", backtest.predict.sum())
diff_percent = (backtest.doanhthu.sum() - backtest.predict.sum())/backtest.doanhthu.sum()
print("diff_percent", diff_percent)
print("mape", mean_absolute_percentage_error(backtest.doanhthu, backtest.predict))
print("rmse", mean_squared_error(backtest.doanhthu, backtest.predict, squared=False))

Doanh thu thuc te 2956154546.4
Doanh thu du doan 2974140761.8630157
diff_percent -0.006084328535840299
mape 12.61429884106692
rmse 12496922.262735879


In [805]:
backtest_11["predict"] = predict_model(all_data_10, backtest_11,best_params, "doanhthu")
(backtest_11.doanhthu.sum() - backtest_11.predict.sum())/backtest_11.doanhthu.sum()

-0.03564052867201861

In [806]:
print("Doanh thu thuc te", backtest_11.doanhthu.sum())
print("Doanh thu du doan", backtest_11.predict.sum())
diff_percent = (backtest_11.doanhthu.sum() - backtest_11.predict.sum())/backtest_11.doanhthu.sum()
print("diff_percent", diff_percent)
print("mape", mean_absolute_percentage_error(backtest_11.doanhthu, backtest_11.predict))
print("rmse", mean_squared_error(backtest_11.doanhthu, backtest_11.predict, squared=False))

Doanh thu thuc te 1868598063.5
Doanh thu du doan 1935195886.3586502
diff_percent -0.03564052867201861
mape 10.612364249361809
rmse 11374600.910552347


In [807]:
fig = px.line(backtest_11, x="ngay", y=["doanhthu","predict"])
fig.show()

In [808]:
trannao_10 = backtest[["ngay", "predict"]]
trannao_10.columns = ["ngay", "trannao"]
trannao_20_11 = backtest_11[["ngay", "predict"]]
trannao_20_11.columns = ["ngay", "trannao"]

In [809]:
trannao_12 = predict_next(data_df,best_params,"doanhthu",41)

In [810]:
trannao_12 = trannao_12[["ds", "predict"]]
trannao_12.columns = ["ds", "trannao"]

## nguyentriphuong

In [813]:
nguyentriphuong = store[store.macuahang==19]

In [814]:
tranao

,macuahang,tencuahang,ngay,soluonghoadon,soluongkhach,doanhso,giamgiamon,giamgiahoadon,giamgiathanhvien,vat,phuphi,doanhthu,dow
2329,19,ICool Nguy?n Tri Ph??ng,2018-02-04,29,141,23819700.0,349000.0,15000.0,296680.0,0.00,0,23159020.0,6
2345,19,ICool Nguy?n Tri Ph??ng,2018-02-05,27,110,13179900.0,0.0,0.0,115005.0,0.00,0,13064895.0,0
2361,19,ICool Nguy?n Tri Ph??ng,2018-02-06,30,136,19341700.0,500000.0,0.0,134680.0,178160.00,0,18707020.0,1
2377,19,ICool Nguy?n Tri Ph??ng,2018-02-07,20,93,13963300.0,376000.0,124910.0,58670.0,0.00,150000,13403720.0,2
2393,19,ICool Nguy?n Tri Ph??ng,2018-02-08,37,181,23196800.0,1044000.0,120870.0,331470.0,0.00,0,21700460.0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
579,19,ICool Nguy?n Tri Ph??ng,2020-10-20,23,110,18904400.0,250000.0,282150.0,0.0,0.00,0,18372250.0,1
580,19,ICool Nguy?n Tri Ph??ng,2020-10-21,16,80,11296800.0,0.0,0.0,0.0,0.00,0,11296800.0,2
581,19,ICool Nguy?n Tri Ph??ng,2020-10-22,7,40,6043200.0,186480.0,0.0,0.0,0.00,0,5856720.0,3
582,19,ICool Nguy?n Tri Ph??ng,2020-10-23,14,84,14112800.0,151620.0,113280.0,79625.0,75643.75,0,13768275.0,4


In [815]:
firstday = dt.datetime.strptime('2019/1/1', '%Y/%m/%d').date()
lastday = dt.datetime.strptime('2020/9/30', '%Y/%m/%d').date() 

In [816]:
date_period = pd.date_range(firstday, lastday- dt.timedelta(days = 61))
date_period1 = pd.date_range(lastday- dt.timedelta(days = 60), lastday- dt.timedelta(days = 30))
date_period2 = pd.date_range(lastday- dt.timedelta(days = 29), lastday)

In [817]:
all_prediod =  pd.date_range(firstday, lastday)
date_test_period =  pd.date_range(dt.datetime.strptime('2020/10/1', '%Y/%m/%d').date(), dt.datetime.strptime('2020/10/31', '%Y/%m/%d').date() )
date_11 =  pd.date_range(dt.datetime.strptime('2020/11/1', '%Y/%m/%d').date(), dt.datetime.strptime('2020/11/20', '%Y/%m/%d').date() )

In [818]:
date_df =pd.DataFrame(pd.date_range(firstday, dt.datetime.strptime('2020/11/20', '%Y/%m/%d').date() ), columns = ["ngay"])
data_df = date_df.merge(nguyentriphuong[["ngay", "doanhthu"]], on = "ngay", how = "left")
# date_df.comlumns

In [819]:
data_df["nfl_sunday"] = data_df['ngay'].apply(nfl_sunday)
data_df["nfl_monday"] = data_df['ngay'].apply(nfl_monday)

In [820]:
train = data_df[data_df.ngay.isin(list(date_period)+list(date_period1) + list(date_period2))]
test = data_df[data_df.ngay.isin(date_period2)].reset_index().drop(["index"], axis=1)

In [821]:
all_data = data_df[data_df.ngay.isin(all_prediod)].reset_index().drop(["index"], axis=1)
all_data_10 = data_df[data_df.ngay.isin(pd.date_range(firstday,dt.datetime.strptime('2020/10/31', '%Y/%m/%d').date()))].reset_index().drop(["index"], axis=1)
backtest = data_df[data_df.ngay.isin(date_test_period)].reset_index().drop(["index"], axis=1)
backtest_11 = data_df[data_df.ngay.isin(date_11)].reset_index().drop(["index"], axis=1)

In [822]:
error = onecolfcst("doanhthu", params = params)

 14%|█▍        | 1/7 [00:02<00:14,  2.45s/it]

[3, 5, 10, 0.5, 10, 0.5] 0.489749443374825


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f24554b8dd8>
 29%|██▊       | 2/7 [00:05<00:12,  2.51s/it]

[25, 25, 50, 0.1, 10, 0.1] 0.5201464767780488


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f245530d278>
 43%|████▎     | 3/7 [00:09<00:11,  2.97s/it]

...

[2, 2, 4, 0.8, 10, 0.8] 0.5056066076038046


INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f245530d518>
 86%|████████▌ | 6/7 [00:15<00:02,  2.45s/it]INFO:fbprophet:Applying in parallel with <concurrent.futures.process.ProcessPoolExecutor object at 0x7f245341f7f0>


[2, 3, 25, 0.1, 10, 0.1] 0.5151487090708731


100%|██████████| 7/7 [00:17<00:00,  2.52s/it]

[5, 2, 2, 0.5, 0.3, 0.01] 0.667464181373768


In [868]:
best_params = [3, 3, 2, 0.6, 9, 0.6]

In [871]:
# best_params = error[error.rmse == error.rmse.min()].params.values[0]
backtest["predict"] = predict_model(all_data, backtest,best_params, "doanhthu")

In [872]:
fig = px.line(backtest, x="ngay", y=["doanhthu","predict"])
fig.show()

In [873]:
print("Doanh thu thuc te", backtest.doanhthu.sum())
print("Doanh thu du doan", backtest.predict.sum())
diff_percent = (backtest.doanhthu.sum() - backtest.predict.sum())/backtest.doanhthu.sum()
print("diff_percent", diff_percent)
print("mape", mean_absolute_percentage_error(backtest.doanhthu, backtest.predict))
print("rmse", mean_squared_error(backtest.doanhthu, backtest.predict, squared=False))

Doanh thu thuc te 485317722.0
Doanh thu du doan 485860595.776762
diff_percent -0.0011185945869127125
mape 53.57304704381259
rmse 5806124.342198839


In [874]:
backtest_11["predict"] = predict_model(all_data_10, backtest_11,best_params, "doanhthu")
(backtest_11.doanhthu.sum() - backtest_11.predict.sum())/backtest_11.doanhthu.sum()

-0.09801392231591469

In [875]:
print("Doanh thu thuc te", backtest_11.doanhthu.sum())
print("Doanh thu du doan", backtest_11.predict.sum())
diff_percent = (backtest_11.doanhthu.sum() - backtest_11.predict.sum())/backtest_11.doanhthu.sum()
print("diff_percent", diff_percent)
print("mape", mean_absolute_percentage_error(backtest_11.doanhthu, backtest_11.predict))
print("rmse", mean_squared_error(backtest_11.doanhthu, backtest_11.predict, squared=False))

Doanh thu thuc te 261686255.0
Doanh thu du doan 287335151.26871264
diff_percent -0.09801392231591469
mape 40.90824062309714
rmse 4726468.473533656


In [876]:
fig = px.line(backtest_11, x="ngay", y=["doanhthu","predict"])
fig.show()

In [877]:
nguyentriphuong_10 = backtest[["ngay", "predict"]]
nguyentriphuong_10.columns = ["ngay", "nguyentriphuong"]
nguyentriphuong_20_11 = backtest_11[["ngay", "predict"]]
nguyentriphuong_20_11.columns = ["ngay", "nguyentriphuong"]

In [878]:
nguyentriphuong_12 = predict_next(data_df,best_params,"doanhthu",41)

In [879]:
nguyentriphuong_12 = nguyentriphuong_12[["ds", "predict"]]
nguyentriphuong_12.columns = ["ds", "nguyentriphuong"]

In [881]:
total = ungvankhiem_12.merge(xvnt_12, on = "ds").merge(toky_12, on = "ds").merge(binhphu_12,on ="ds").merge(cauchuY_12,on = "ds").merge(trannao_12, on ="ds").merge(nguyentriphuong_12, on ="ds")

In [883]:
total.to_csv("revenue_12_new.csv", index=False)

In [886]:
total_10 = ungvankhiem_10.merge(xvnt_10, on = "ngay").merge(toky_10, on = "ngay").merge(binhphu_10,on ="ngay").merge(cauchuY_10,on = "ngay").merge(trannao_10, on ="ngay").merge(nguyentriphuong_10, on ="ngay")

In [888]:
total_10.to_csv("revenue_10_new.csv", index=False)

In [889]:
total_20_11 = ungvankhiem_20_11.merge(xvnt_20_11, on = "ngay").merge(toky_20_11, on = "ngay").merge(binhphu_20_11,on ="ngay").merge(cauchuY_20_11,on = "ngay").merge(trannao_20_11, on ="ngay").merge(nguyentriphuong_20_11, on ="ngay")

In [890]:
total_20_11.to_csv("revenue_20_11_new.csv", index=False)